In [ ]:
# 1. Install Libraries
!pip install -q transformers accelerate scikit-learn

import os
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive

# Fix seed for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [ ]:
#cell 2

from huggingface_hub import login
login()


In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
model.eval()

# Punctuation ID Mapping
PUNCT_MAP = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1]
}
EOS_ID = tokenizer.eos_token_id
print(f"Punctuation IDs: {PUNCT_MAP}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Punctuation IDs: {'COMMA': 11, 'PERIOD': 13, 'QMARK': 30}


In [ ]:
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. User-provided path and load code
BASE_PATH = "/content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2"
VAL_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_validation.Y.txt")
TEST_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_test.Y.txt")

assert os.path.exists(VAL_Y_PATH), f"Not found: {VAL_Y_PATH}"

print(f"Loading file: {VAL_Y_PATH}")
with open(VAL_Y_PATH, "r", encoding="utf-8") as f:
    val_y_list = [line.strip() for line in f if line.strip()]

with open(TEST_Y_PATH, "r", encoding="utf-8") as f:
    test_y_list = [line.strip() for line in f if line.strip()]

print("num val samples:", len(val_y_list))
print("example:", val_y_list[0])

Mounted at /content/drive
Loading file: /content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2/iwslt2017_en_validation.Y.txt
num val samples: 1501
example: Last year I showed these two slides so that demonstrate that the arctic ice cap, which for most of the last three million years has been the size of the lower 48 states, has shrunk by 40 percent.


In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score
import re

# --- 1. Configuration ---
# Use entire section to secure question mark (QMARK) samples
dataset_subset = val_y_list
CALIB_SIZE = len(dataset_subset)
K_VALUE = 4  # K=4 Tokens (Strict)

# Punctuation Token IDs
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())

def parse_sentence_to_boundaries(text: str):
    tokens = text.strip().split()
    boundaries = []
    for tok in tokens:
        # Pattern: (content)(punctuation)(zero or more closing quotes/parentheses/brackets)
        m = re.match(r'^(.*?)([,.?])(["\'
)\]\}]*)$', tok)

        if m:
            base, punct_char = m.group(1), m.group(2)

            if punct_char == ',':
                label = "COMMA"
            elif punct_char == '.':
                label = "PERIOD"
            elif punct_char == '?':
                label = "QMARK"

            word = base
        else:
            label = "O"
            word = tok

        # Remove quotes/parentheses from words (minimal required)
        word = re.sub(r'[\"\'
(\)\[\]\{\}]', '', word)

        # Do not treat tokens like just quotes (") as boundaries
        if word:
            boundaries.append({"word": word, "label": label})
            continue

        # Case 2: Punctuation-only tokens without words (e.g., '."', ',"', '?')
        # This punctuation belongs to the label of the preceding word
        if m and boundaries:
            # If the previous label is already punctuation, a policy is needed whether to overwrite/maintain it
            # Usually, the last punctuation mark is stronger, so overwriting is recommended
            boundaries[-1]["label"] = label

        # Case 3: Ignore tokens like just quotes (")
        # (If neither m nor word is empty, it comes here)

    return boundaries

# Helper function: Calculate joint probability using chunking
def get_joint_score_optimized(prefix_ids, target_ids):
    """
    prefix_ids: Context tokens so far
    target_ids: Next K tokens (lookahead_tokens)
    """
    if not target_ids: return 0.0
    full_input = torch.tensor([prefix_ids + target_ids], device=device)
    with torch.no_grad():
        out = model(full_input)

    start_pos = len(prefix_ids) - 1
    # Extract Logits corresponding to the position of the next K tokens
    rel_logits = out.logits[0, start_pos : start_pos + len(target_ids), :]
    log_probs = torch.log_softmax(rel_logits, dim=-1)
    t_ids_tensor = torch.tensor(target_ids, device=device)

    # Return the joint probability of those tokens
    return log_probs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

In [ ]:
# [Cell: K=1 Calibration with Strict Logic]
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score
import re

# --- 1. 설정 ---
# 데이터 분포 균형을 위해 K=4와 동일한 구간 사용
dataset_subset = val_y_list
K_VALUE = 1  # K=1 Token (Strict)

# 구두점 토큰 ID
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())


# --- 2. 데이터 수집 ---
print(f"Step 1: Collecting scores for K={K_VALUE} (Strict Token Logic)...")
raw_data = []
EOS_ID = tokenizer.eos_token_id

for sent_idx, y_true in enumerate(tqdm(dataset_subset)):
    tokens = y_true.strip().split()
    boundaries = []
    for tok in tokens:
        word = tok
        label = "O"
        if tok.endswith(","): label = "COMMA"; word = tok[:-1]
        elif tok.endswith("."): label = "PERIOD"; word = tok[:-1]
        elif tok.endswith("?"): label = "QMARK"; word = tok[:-1]
        word = re.sub(r'[\"\'\(\)]', '', word)
        if word: boundaries.append({"word": word, "label": label})

    if not boundaries: continue

    current_words = []
    for i, item in enumerate(boundaries):
        word = item['word']
        gold_label = item['label']
        current_words.append(word)

        history_text = " ".join(current_words)
        h_ids = tokenizer.encode(history_text, add_special_tokens=True)

        # [Safety] EOS 제거
        if len(h_ids) > 0 and h_ids[-1] == EOS_ID:
            h_ids = h_ids[:-1]

        # [Strict K=1 Lookahead]
        # 미래 전체를 가져와서 토큰화 후, 맨 앞 1개만 자름 (BPE 문맥 보존)
        full_future_words = [b['word'] for b in boundaries[i+1:]]
        full_future_str = " ".join(full_future_words)

        if not full_future_str:
            lookahead_tokens = [EOS_ID] * K_VALUE # [EOS]
        else:
            full_future_ids = tokenizer.encode(" " + full_future_str, add_special_tokens=False)
            lookahead_tokens = full_future_ids[:K_VALUE] # 앞에서 1개

            # Padding (만약 토큰이 없으면 EOS)
            if len(lookahead_tokens) < K_VALUE:
                lookahead_tokens = lookahead_tokens + ([EOS_ID] * (K_VALUE - len(lookahead_tokens)))

        # S0, Cost, Gain 계산
        s0 = get_joint_score_optimized(h_ids, lookahead_tokens)

        with torch.no_grad():
            h_out = model(torch.tensor([h_ids], device=device))
        base_logprobs = torch.log_softmax(h_out.logits[0, -1, :], dim=-1)

        batch_input_ids = [h_ids + [pid] + lookahead_tokens for _, pid in PUNCT_LIST]
        batch_tensor = torch.tensor(batch_input_ids, device=device)
        with torch.no_grad():
            batch_out = model(batch_tensor)

        entry = {"gold": gold_label}
        for b_idx, (pname, pid) in enumerate(PUNCT_LIST):
            entry[f"{pname}_cost"] = base_logprobs[pid].item()

            start_pos = len(h_ids)
            rel_logits = batch_out.logits[b_idx, start_pos : start_pos + len(lookahead_tokens), :]
            lprobs = torch.log_softmax(rel_logits, dim=-1)
            t_ids_tensor = torch.tensor(lookahead_tokens, device=device)
            p_joint_score = lprobs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

            entry[f"{pname}_gain"] = p_joint_score - s0

        raw_data.append(entry)

        if gold_label != "O":
            punct_char = "," if gold_label == "COMMA" else ("." if gold_label == "PERIOD" else "?")
            current_words[-1] = current_words[-1] + punct_char

df_scores = pd.DataFrame(raw_data)

# --- 3. 그리드 서치 ---
print("\nStep 2: Searching for optimal parameters...")
alpha_range = np.arange(0.1, 0.95, 0.05)
threshold_range = np.arange(-3.0, 2.0, 0.25)

best_score = -1
best_params = {}
golds = df_scores["gold"].values
EVAL_LABELS = ["COMMA", "PERIOD", "QMARK"] # 'O' 제외

for alpha in alpha_range:
    for thresh in threshold_range:
        preds = []
        for _, row in df_scores.iterrows():
            best_p, max_s = "O", float("-inf")
            for pname in ["COMMA", "PERIOD", "QMARK"]:
                score = (alpha * row[f"{pname}_cost"]) + ((1 - alpha) * row[f"{pname}_gain"])
                if score > max_s:
                    max_s, best_p = score, pname
            if max_s <= thresh: best_p = "O"
            preds.append(best_p)

        current_score = f1_score(golds, preds, average="macro", labels=EVAL_LABELS, zero_division=0)

        if current_score > best_score:
            best_score, best_params = current_score, {"alpha": alpha, "threshold": thresh}

print(f"\n=== K={K_VALUE} (Strict Token) Optimization Results ===")
print(f"Best Macro F1 (excl. O): {best_score:.4f}")
print(f"Optimal ALPHA: {best_params['alpha']:.2f}")
print(f"Optimal THRESHOLD: {best_params['threshold']:.2f}")

Step 1: Collecting scores for K=1 (Strict Token Logic)...


100%|██████████| 1501/1501 [29:30<00:00,  1.18s/it]



Step 2: Searching for optimal parameters...

=== K=1 (Strict Token) Optimization Results ===
Best Macro F1 (excl. O): 0.8714
Optimal ALPHA: 0.55
Optimal THRESHOLD: -1.00


In [ ]:
# [Cell: K=1 Calibration with Strict Logic]
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score
import re

# --- 1. Configuration ---
# Use the same section as K=4 for data distribution balance
dataset_subset = val_y_list
K_VALUE = 1  # K=1 Token (Strict)

# Punctuation Token IDs
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())


# --- 2. Data Collection ---
print(f"Step 1: Collecting scores for K={K_VALUE} (Strict Token Logic)...")
raw_data = []
EOS_ID = tokenizer.eos_token_id

for sent_idx, y_true in enumerate(tqdm(dataset_subset)):
    tokens = y_true.strip().split()
    boundaries = []
    for tok in tokens:
        word = tok
        label = "O"
        if tok.endswith(","): label = "COMMA"; word = tok[:-1]
        elif tok.endswith("."): label = "PERIOD"; word = tok[:-1]
        elif tok.endswith("?"): label = "QMARK"; word = tok[:-1]
        word = re.sub(r'[\"\'
(\)]', '', word)
        if word: boundaries.append({"word": word, "label": label})

    if not boundaries: continue

    current_words = []
    for i, item in enumerate(boundaries):
        word = item['word']
        gold_label = item['label']
        current_words.append(word)

        history_text = " ".join(current_words)
        h_ids = tokenizer.encode(history_text, add_special_tokens=True)

        # [Safety] Remove EOS
        if len(h_ids) > 0 and h_ids[-1] == EOS_ID:
            h_ids = h_ids[:-1]

        # [Strict K=1 Lookahead]
        # Tokenize the entire future and then cut off the first 1 (BPE context preservation)
        full_future_words = [b['word'] for b in boundaries[i+1:]]
        full_future_str = " ".join(full_future_words)

        if not full_future_str:
            lookahead_tokens = [EOS_ID] * K_VALUE # [EOS]
        else:
            full_future_ids = tokenizer.encode(" " + full_future_str, add_special_tokens=False)
            lookahead_tokens = full_future_ids[:K_VALUE] # Take the first 1

            # Padding (EOS if no tokens)
            if len(lookahead_tokens) < K_VALUE:
                lookahead_tokens = lookahead_tokens + ([EOS_ID] * (K_VALUE - len(lookahead_tokens)))

        # Calculate S0, Cost, Gain
        s0 = get_joint_score_optimized(h_ids, lookahead_tokens)

        with torch.no_grad():
            h_out = model(torch.tensor([h_ids], device=device))
        base_logprobs = torch.log_softmax(h_out.logits[0, -1, :], dim=-1)

        batch_input_ids = [h_ids + [pid] + lookahead_tokens for _, pid in PUNCT_LIST]
        batch_tensor = torch.tensor(batch_input_ids, device=device)
        with torch.no_grad():
            batch_out = model(batch_tensor)

        entry = {"gold": gold_label}
        for b_idx, (pname, pid) in enumerate(PUNCT_LIST):
            entry[f"{pname}_cost"] = base_logprobs[pid].item()

            start_pos = len(h_ids)
            rel_logits = batch_out.logits[b_idx, start_pos : start_pos + len(lookahead_tokens), :]
            lprobs = torch.log_softmax(rel_logits, dim=-1)
            t_ids_tensor = torch.tensor(lookahead_tokens, device=device)
            p_joint_score = lprobs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

            entry[f"{pname}_gain"] = p_joint_score - s0

        raw_data.append(entry)

        if gold_label != "O":
            punct_char = "," if gold_label == "COMMA" else ("." if gold_label == "PERIOD" else "?")
            current_words[-1] = current_words[-1] + punct_char

df_scores = pd.DataFrame(raw_data)

# --- 3. Grid Search ---
print("\nStep 2: Searching for optimal parameters...")
alpha_range = np.arange(0.1, 0.95, 0.05)
threshold_range = np.arange(-3.0, 2.0, 0.25)

best_score = -1
best_params = {}
golds = df_scores["gold"].values
EVAL_LABELS = ["COMMA", "PERIOD", "QMARK"] # Exclude 'O'

for alpha in alpha_range:
    for thresh in threshold_range:
        preds = []
        for _, row in df_scores.iterrows():
            best_p, max_s = "O", float("-inf")
            for pname in ["COMMA", "PERIOD", "QMARK"]:
                score = (alpha * row[f"{pname}_cost"]) + ((1 - alpha) * row[f"{pname}_gain"])
                if score > max_s:
                    max_s, best_p = score, pname
            if max_s <= thresh: best_p = "O"
            preds.append(best_p)

        current_score = f1_score(golds, preds, average="macro", labels=EVAL_LABELS, zero_division=0)

        if current_score > best_score:
            best_score, best_params = current_score, {"alpha": alpha, "threshold": thresh}

print(f"\n=== K={K_VALUE} (Strict Token) Optimization Results ===")
print(f"Best Macro F1 (excl. O): {best_score:.4f}")
print(f"Optimal ALPHA: {best_params['alpha']:.2f}")
print(f"Optimal THRESHOLD: {best_params['threshold']:.2f}")

In [ ]:
# [Cell: Final Inference for K=1 | Strict Logic + Optimized Speed | Metric: Words/s]
import torch
import pandas as pd
import time
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import re
import numpy as np

# --- 1. Apply Optimized Parameters (K=1 Calibration Results) ---
ALPHA_K1 = 0.55          # Optimal Alpha
THRESHOLD_K1 = -1.00     # Optimal Threshold
K_VALUE = 1              # Strict K=1

# Output Order
LABELS_ORDER = ["O", "COMMA", "PERIOD", "QMARK"]

# Punctuation Token IDs
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())

# --- 2. Evaluation Loop ---
print(f"Starting Final Evaluation (K={K_VALUE}, Strict Token Logic, Optimized Speed)")
print(f"Params: Alpha={ALPHA_K1}, Threshold={THRESHOLD_K1}")
print("Metric: Words/second")

all_golds = []
all_preds = []
total_processed_words = 0  # Word unit count
start_time = time.time()
EOS_ID = tokenizer.eos_token_id

for sent_idx, y_true in enumerate(tqdm(test_y_list)):
    boundaries = parse_sentence_to_boundaries(y_true)
    if not boundaries: continue

    current_words = []
    for i, item in enumerate(boundaries):
        word, gold_label = item['word'], item['label']
        current_words.append(word)

        # Increment processed word count
        total_processed_words += 1

        history_text = " ".join(current_words)
        h_ids = tokenizer.encode(history_text, add_special_tokens=True)

        # [Safety] Remove EOS
        if len(h_ids) > 0 and h_ids[-1] == EOS_ID:
            h_ids = h_ids[:-1]

        # [Strict K=1 Lookahead]
        full_future_words = [b['word'] for b in boundaries[i+1:]]
        full_future_str = " ".join(full_future_words)

        if not full_future_str:
            lookahead_tokens = [EOS_ID] * K_VALUE
        else:
            full_future_ids = tokenizer.encode(" " + full_future_str, add_special_tokens=False)
            lookahead_tokens = full_future_ids[:K_VALUE]

            if len(lookahead_tokens) < K_VALUE:
                lookahead_tokens = lookahead_tokens + ([EOS_ID] * (K_VALUE - len(lookahead_tokens)))

        target_token_id = lookahead_tokens[0]

        # 1. Base Forward (Cost + s0 Optimized Extraction)
        with torch.no_grad():
            h_out = model(torch.tensor([h_ids], device=device))
        base_logprobs = torch.log_softmax(h_out.logits[0, -1, :], dim=-1)

        s0 = base_logprobs[target_token_id].item()

        # 2. Batch Forward (Gain)
        batch_input_ids = [h_ids + [pid] + lookahead_tokens for _, pid in PUNCT_LIST]
        batch_tensor = torch.tensor(batch_input_ids, device=device)
        with torch.no_grad():
            batch_out = model(batch_tensor)

        best_p, max_s = "O", float('-inf')
        for b_idx, (pname, pid) in enumerate(PUNCT_LIST):
            cost = base_logprobs[pid].item()

            # Extract Gain
            start_pos = len(h_ids)
            rel_logits = batch_out.logits[b_idx, start_pos : start_pos + len(lookahead_tokens), :]
            lprobs = torch.log_softmax(rel_logits, dim=-1)
            t_ids_tensor = torch.tensor(lookahead_tokens, device=device)
            p_joint_score = lprobs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

            gain = p_joint_score - s0

            score = (ALPHA_K1 * cost) + ((1 - ALPHA_K1) * gain)

            if score > THRESHOLD_K1 and score > max_s:
                max_s, best_p = score, pname

        all_golds.append(gold_label)
        all_preds.append(best_p)

        if best_p != "O":
            punct_char = "," if best_p == "COMMA" else ("." if best_p == "PERIOD" else "?")
            current_words[-1] = current_words[-1] + punct_char

# --- 3. Results Report ---
end_time = time.time()
elapsed = end_time - start_time
wps = total_processed_words / elapsed  # Words Per Second

print(f"\n[Final Results | K={K_VALUE} Strict Optimized]")
print(f"Inference Speed: {wps:.2f} words/s")
print(f"Total Processed Words: {total_processed_words}")
print(f"Total Execution Time: {elapsed:.2f}s")
print("-" * 60)
print(classification_report(all_golds, all_preds, labels=LABELS_ORDER, zero_division=0, digits=3))

print("\nConfusion Matrix")
cm = confusion_matrix(all_golds, all_preds, labels=LABELS_ORDER)
df_cm = pd.DataFrame(cm, index=[f"True_{l}" for l in LABELS_ORDER], columns=[f"Pred_{l}" for l in LABELS_ORDER])
print(df_cm)

Starting Final Evaluation (K=1, Strict Token Logic, Optimized Speed)
Params: Alpha=0.55, Threshold=-1.0
Metric: Words/second


100%|██████████| 10799/10799 [1:57:59<00:00,  1.53it/s]



[Final Results | K=1 Strict Optimized]
Inference Speed: 26.03 words/s
Total Processed Words: 184278
Total Execution Time: 7079.07s
------------------------------------------------------------
              precision    recall  f1-score   support

           O      0.984     0.971     0.977    160196
       COMMA      0.699     0.815     0.753     13017
      PERIOD      0.949     0.942     0.946     10153
       QMARK      0.848     0.888     0.868       912

    accuracy                          0.958    184278
   macro avg      0.870     0.904     0.886    184278
weighted avg      0.961     0.958     0.959    184278


Confusion Matrix
             Pred_O  Pred_COMMA  Pred_PERIOD  Pred_QMARK
True_O       155471        4485          217          23
True_COMMA     2171       10613          221          12
True_PERIOD     396          79         9568         110
True_QMARK       20           7           75         810
